# Neural Machine Translation (English ↔ Hindi): Kaggle training

Trains the custom Transformer (`ml/transformer/`) on the full IIT Bombay corpus, **both directions at once** (one GPU each on *GPU T4 x2*), then evaluates on the official test split with BLEU and chrF++.

### One-time setup
1. **Settings → Accelerator → GPU T4 x2**, and **Settings → Internet → On**.
   (Your Kaggle account must be phone-verified to use GPUs and internet. Do **not** pick P100: recent PyTorch builds on Kaggle no longer support it.)
2. Get the code in (the notebook detects which you used):
   - **Upload (recommended):** on your PC run `python scripts/make_kaggle_bundle.py`, upload `dist/nmt_code.zip` as a new Kaggle dataset, then **Add Input** that dataset here. It already contains the trained tokenizers.
   - **GitHub:** push the repo and set `REPO_URL` below; used only when no uploaded code is attached.

### Each training session (about 12 h)
- Click **Save Version → Save & Run All (Commit)**. It runs in the background, so you can close the browser.
- Everything is written to `/kaggle/working/nmt_artifacts` (Output tab).

### Continue training in the next session
- **Add Input → Notebook Output Files → this notebook** (the latest version), then **Save & Run All** again.
- The notebook finds `nmt_artifacts`, restores the **same tokenizers** and `last.pt`, and resumes mid-epoch.
- **Never** retrain tokenizers between sessions: a checkpoint only works with the tokenizer it was trained with.

### When finished
- Download `nmt_artifacts/models/<run>/` (contains `best.pt`, `spm_en.model`, `spm_hi.model`, `metrics.json`, eval reports) into the repo's `models/` folder on your PC.

In [ ]:
# ======================= CONFIGURATION =======================
REPO_URL = "https://github.com/yadavlakshya743/Neural-Machine-Translation-System.git"
REPO_BRANCH = "main"
# The attached nmt_code dataset is used automatically if present; otherwise REPO_URL is cloned.

CONFIG = "ml/configs/transformer_base.yaml"
DIRECTIONS = ["en-hi", "hi-en"]    # trained in parallel when 2 GPUs are available

SESSION_HOURS = 12.0               # Kaggle hard limit per session
RESERVE_HOURS = 0.75               # kept for evaluation + saving outputs
# ==============================================================
import os, sys, time, glob, json, shutil, subprocess
NOTEBOOK_START = time.time()
WORK, CODE = "/kaggle/working", "/tmp/nmt"     # code lives outside /kaggle/working so outputs stay small
OUT = f"{WORK}/nmt_artifacts"
!nvidia-smi --query-gpu=index,name,memory.total --format=csv

In [ ]:
# ---- Get the code
if os.path.exists(CODE):
    shutil.rmtree(CODE)
hits = glob.glob("/kaggle/input/**/ml/transformer/model.py", recursive=True)
if hits:
    src_root = os.path.dirname(os.path.dirname(os.path.dirname(hits[0])))
    print("Using uploaded code from", src_root)
    shutil.copytree(src_root, CODE)
else:
    print("No uploaded code attached: cloning", REPO_URL)
    subprocess.run(["git", "clone", "--depth", "1", "-b", REPO_BRANCH, REPO_URL, CODE], check=True)
os.chdir(CODE)
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "sentencepiece>=0.1.99", "sacrebleu>=2.3.1", "pyyaml", "pyarrow"], check=True)
import yaml
MODEL_NAME = yaml.safe_load(open(CONFIG))["model"]["name"]
print("Code ready at", CODE, "| model", MODEL_NAME)

In [ ]:
# ---- Restore tokenizers + checkpoints from a previous session (if attached as input)
prev = sorted(p for p in glob.glob("/kaggle/input/**/nmt_artifacts", recursive=True) if os.path.isdir(p))
if prev:
    prev = prev[0]
    print("Resuming from previous output:", prev)
    os.makedirs("ml/tokenization", exist_ok=True)
    for lang in ("en", "hi"):
        shutil.copy2(f"{prev}/tokenizers/spm_{lang}.model", f"ml/tokenization/spm_{lang}.model")
    if os.path.isdir(f"{prev}/models"):
        shutil.copytree(f"{prev}/models", "models", dirs_exist_ok=True)
    !ls -la models/*/
else:
    print("No previous output attached: fresh start.")

In [ ]:
# ---- Download + clean the corpus (existing tokenizers are reused, never retrained)
!python scripts/prepare_data.py
print(open("data/processed/data_report.json").read())

In [ ]:
# ---- Train (both directions in parallel when 2 GPUs are present)
import torch

def hours_left():
    return SESSION_HOURS - (time.time() - NOTEBOOK_START) / 3600 - RESERVE_HOURS

def launch(direction, gpu, hours):
    log_path = f"{WORK}/train_{direction}.log"
    env = {**os.environ, "CUDA_VISIBLE_DEVICES": str(gpu), "PYTHONUNBUFFERED": "1"}
    cmd = [sys.executable, "scripts/train.py", "--config", CONFIG, "--direction", direction,
           "--resume", "auto", "--max_hours", f"{hours:.2f}"]
    print(f"[{direction}] GPU {gpu}, budget {hours:.2f} h -> {log_path}")
    return direction, subprocess.Popen(cmd, stdout=open(log_path, "a"), stderr=subprocess.STDOUT, env=env), log_path

def wait(jobs, every_sec=900):
    while any(p.poll() is None for _, p, _ in jobs):
        time.sleep(every_sec)
        for d, p, log in jobs:
            tail = open(log, encoding="utf-8", errors="replace").read().splitlines()[-2:]
            print(f"[{d}] {'running' if p.poll() is None else 'exit ' + str(p.returncode)} | " + " || ".join(tail), flush=True)
    for d, p, log in jobs:
        print(f"[{d}] finished with exit code {p.returncode}")
        if p.returncode != 0:
            print(open(log, encoding="utf-8", errors="replace").read()[-3000:])

n_gpu = torch.cuda.device_count()
assert n_gpu > 0, "No GPU: enable a GPU accelerator in Settings."
if n_gpu >= len(DIRECTIONS):
    wait([launch(d, i, hours_left()) for i, d in enumerate(DIRECTIONS)])
else:
    print(f"Only {n_gpu} GPU: training directions one after another.")
    for k, d in enumerate(DIRECTIONS):
        wait([launch(d, 0, hours_left() / (len(DIRECTIONS) - k))])

In [ ]:
# ---- Evaluate best checkpoints on the official test split (2,507 sentences)
for d in DIRECTIONS:
    ckpt = f"models/{MODEL_NAME}_{d}/best.pt"
    if not os.path.exists(ckpt):
        print("No best.pt yet for", d)
        continue
    !python scripts/evaluate.py --checkpoint {ckpt} --strategy greedy
    !python scripts/evaluate.py --checkpoint {ckpt} --strategy beam_search --beam_size 4

In [ ]:
# ---- Save everything needed to resume or deploy into /kaggle/working/nmt_artifacts
os.makedirs(f"{OUT}/tokenizers", exist_ok=True)
for lang in ("en", "hi"):
    shutil.copy2(f"ml/tokenization/spm_{lang}.model", f"{OUT}/tokenizers/spm_{lang}.model")
if os.path.isdir("models"):
    shutil.copytree("models", f"{OUT}/models", dirs_exist_ok=True)
os.makedirs(f"{OUT}/data", exist_ok=True)
for f in ("data_report.json", "manifest.json"):
    if os.path.exists(f"data/processed/{f}"):
        shutil.copy2(f"data/processed/{f}", f"{OUT}/data/{f}")
for log in glob.glob(f"{WORK}/train_*.log"):
    shutil.copy2(log, OUT)

for d in DIRECTIONS:
    m = f"{OUT}/models/{MODEL_NAME}_{d}/metrics.json"
    if os.path.exists(m):
        st = json.load(open(m))["state"]
        print(f"{d}: status {st['status']} | next epoch {st['epoch']} | step {st['global_step']} | "
              f"best dev BLEU {st['best_score']} (epoch {st['best_epoch']}) | {st['elapsed_sec']/3600:.1f} h total")
        for e in st["history"]["epochs"]:
            print("   ", e)
    for r in sorted(glob.glob(f"{OUT}/models/{MODEL_NAME}_{d}/eval_test_*.json")):
        rep = json.load(open(r))
        print(f"    TEST {rep['decoding']['strategy']}: BLEU {rep['metrics']['bleu']} | chrF++ {rep['metrics']['chrf2']}")
!du -sh {OUT}